In [ ]:
# HPLC-UV/Vis Analysis Code

# Analysis code written by Alex Deans-Rowe

# Setup

import glob
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
#from sys import platform
#from dateutil import parser
import fnmatch as fn
from scipy.signal import find_peaks

#from sklearn.cluster import DBSCAN
#from sklearn.preprocessing import StandardScaler #Normalizes each channel by mean and std deviation
#from sklearn.cluster import KMeans


# Function definitions
def sort_kavita_files(file):
    if fn.fnmatch(file, "*DDW*"):
        key = 9999999999999
    else:
        key = int(file.split(".CSV")[0].split("C")[-1])
    return key

def Plot_Many_Datasets(datadir, datasets, wavelength, file_identifier_strings, faux_color_cycle):

    
    
    print(faux_color_cycle)
    #faux_color_cycle = rainbow_colormap([0:len(datasets)])
    for i,dataset in enumerate(datasets):
        Plot_One_Dataset_Counts(datadir=datadir, dataset=dataset, wavelength=wavelength, file_identifier_strings = file_identifier_strings, plot_color = faux_color_cycle[i])
    return


def Plot_One_Dataset_HPLC(datadir, dataset, wavelength, file_identifier_strings, faux_color_cycle):

    raw_data_series, metadata_series = Get_One_Dataset(datadir=datadir, dataset=dataset, wavelength=wavelength, file_identifier_strings = file_identifier_strings) #, plot_color = faux_color_cycle[i])
    data_series_subtracted = Do_Water_Subtraction(raw_data_series=raw_data_series, file_identifier_strings = file_identifier_strings)
    series_peak_lists, series_peak_counts = Get_Dataset_Peaks(data_series_subtracted=data_series_subtracted, file_identifier_strings=file_identifier_strings)

    global fig, ax

    faux_linestyle_cycle = ['--', '-', ':', '-.']

    
    #print(ax)

    print(series_peak_lists)

    for i, this_experimental_condition in enumerate(data_series_subtracted):
        for j,this_replicate in enumerate(this_experimental_condition):
            for k,this_cycle in enumerate(this_replicate):
                 if k < len(data_series_subtracted[i][j]) - 1: # ignore the water run at the end of the list
                     ax[i].plot(this_cycle["RT"], this_cycle["Absorbance"]+(100*k), color = faux_color_cycle[k], label = (dataset) + ": " + datasets_dict[dataset])


                     
                     
                     ax[i].scatter([this_cycle["RT"][x]+0.1 for x in series_peak_lists[i][j][k]], [this_cycle["Absorbance"][x]+(100*k)+10 for x in series_peak_lists[i][j][k]], marker=mpl.markers.MarkerStyle('<').rotated(deg=45), s=96, color=tuple(x/1.5 for x in faux_color_cycle[k]))
                     ax[i].set_title(file_identifier_strings[i], fontsize=32)
                     #print(f'Cycle {k}: {this_cycle}')
            #ax.scatter(range(len(this_replicate)), this_replicate, color = plot_color, label = (dataset) + ": " + datasets_dict[dataset])
            #ax.plot(range(len(this_replicate)), this_replicate, linestyle = faux_linestyle_cycle[i % 4], color=plot_color)

    #print(data_series_subtracted[0][0][0]["Absorbance"])
    #ax.plot(data_series_subtracted[0][0][0]["RT"],data_series_subtracted[0][0][0]["Absorbance"])
    #for this_series in data_series_subtracted:
    #    ax.plot(this_series)
    return


def Plot_One_Dataset_Counts(datadir, dataset, wavelength, file_identifier_strings, plot_color):
    #full_datapath = os.path.join(datadir, dataset)
    #full_datapath = os.path.join(full_datapath, wavelength)
    #control_datapath = os.path.join(full_datapath, control_string)
    #perturb_string = os.path.join(full_datapath, perturb_string)

    #print(full_datapath)
    
    #datafiles = glob.glob(os.path.join(full_datapath, "**/*.csv"), recursive=True)
    
    #datafiles = sorted(datafiles, key = sort_kavita_files)


    #if dataset == "KMSet9a":
    #    for file in datafiles:
    #        print(file)
    
    # Extract data from datafiles and put them into a list of lists of pandas dataframes.
    # Each identifier string has its own list of dataframes associated with it
    #data_value_series = [[]]
    #metadata_series = [[]]   
    #
    #for i,this_identifier_string in enumerate(file_identifier_strings):
    #    #print(this_identifier_string)
    #    data_value_series.append([])
    #    metadata_series.append([])
    #    for file in datafiles:
    #        #print(file)
    #        if not fn.fnmatch(file, f'*{"rep2"}*'): # Hardcoded way to ignore the fact that KMSet9a and *ONLY* KMSet9a has a second replicate associataed with it
    #            if fn.fnmatch(file, f'*{this_identifier_string}*'):
    #                data_value_series[i].append( pd.read_csv(file, encoding="utf-16", sep="\\t", header=None, engine="python") )
    #                data_value_series[i][-1] = (data_value_series[i][-1]).set_axis(["RT","Absorbance"],axis=1)
    #                metadata_series[i].append( os.path.basename(file) )
    
    #print(len(data_value_series[0]))

    # Subtract water from samples 
    #data_values_subtracted = data_value_series
    #
    #for i,this_identifier_string in enumerate(file_identifier_strings):
    #    for sample in data_values_subtracted[i]:
    #        sample["Absorbance"] = sample["Absorbance"] - data_values_subtracted[i][-1]["Absorbance"]


    # Get counts of peaks for each sample in each series
    #series_peak_counts = []
    #for i,this_identifier_string in enumerate(file_identifier_strings):
    #    #control_peak_counts = []
    #    series_peak_counts.append([])
    #
    #    for j,this_sample in enumerate(data_values_subtracted[i]):
    #    
    #        if j < len(data_values_subtracted[i]) - 1: # ignore the water run at the end of the list
    #            this_peak_list, _ = find_peaks(this_sample["Absorbance"], prominence=15)
    #            series_peak_counts[i].append( len(this_peak_list) )

    raw_data_series, metadata_series = Get_One_Dataset(datadir=datadir, dataset=dataset, wavelength=wavelength, file_identifier_strings = file_identifier_strings) #, plot_color = faux_color_cycle[i])
    data_series_subtracted = Do_Water_Subtraction(raw_data_series=raw_data_series, file_identifier_strings = file_identifier_strings)
    _, series_peak_counts = Get_Dataset_Peaks(data_series_subtracted=data_series_subtracted, file_identifier_strings=file_identifier_strings)



    #for count in series_peak_counts:
    #    print(f'series_peak_counts: {count}')
    
    # Plot the peak lists for each selected series

    global fig, ax
    faux_linestyle_cycle = ['--', '-', ':', '-.']


    for i, this_peak_count_series in enumerate(series_peak_counts):
        for this_replicate in this_peak_count_series:
            print(f'This replicate: {this_replicate}')
            ax.scatter(range(len(this_replicate)), this_replicate, color = plot_color, label = (dataset) + ": " + datasets_dict[dataset])
            ax.plot(range(len(this_replicate)), this_replicate, linestyle = faux_linestyle_cycle[i % 4], color=plot_color)
    
    #for i,this_peak_count_series in enumerate(series_peak_counts):
    #
    #    #print(this_peak_count_series)
    #    print(plot_color)
    #    ax.scatter(range(len(this_peak_count_series)), this_peak_count_series, color=plot_color)
    #    ax.plot(range(len(this_peak_count_series)), this_peak_count_series, linestyle = faux_linestyle_cycle[i % 4], color=plot_color)    
    return


def Plot_Many_Dataset_Ratios(datadir, datasets, wavelength, file_identifier_strings, faux_color_cycle):

  
    
    #print(faux_color_cycle)
    #faux_color_cycle = rainbow_colormap([0:len(datasets)])

    # Get the peak count numbers from each dataset

    dataset_peak_counts = []
    metadata_series = []
    
    for i,dataset in enumerate(datasets):
        raw_data_series, this_metadata_series = Get_One_Dataset(datadir=datadir, dataset=dataset, wavelength=wavelength, file_identifier_strings = file_identifier_strings) #, plot_color = faux_color_cycle[i])
        data_series_subtracted = Do_Water_Subtraction(raw_data_series=raw_data_series, file_identifier_strings = file_identifier_strings)
        _, series_peak_counts = Get_Dataset_Peaks(data_series_subtracted=data_series_subtracted, file_identifier_strings=file_identifier_strings)
        dataset_peak_counts.append(series_peak_counts)
        metadata_series.append(this_metadata_series)
        
    global fig, ax, datasets_dict
    #faux_linestyle_cycle = ['--', '-', ':', '-.']

    for i,series_peak_counts in enumerate(dataset_peak_counts):
        plot_color = faux_color_cycle[i]

        #print("The first series is: ")
        #print(series_peak_counts[0])
        #print("The second series is: ")
        #print(series_peak_counts[1])

        print(series_peak_counts)
        
        this_peak_ratio_series = np.asarray(series_peak_counts[1]) / np.asarray(series_peak_counts[0])
        
        print(this_peak_ratio_series)

        #print(f'The ratio is:\n{this_peak_ratio_series}')
        for this_replicate in this_peak_ratio_series:
            ax.scatter(range(len(this_replicate)), this_replicate, color = plot_color, label = (datasets[i]) + ": " + datasets_dict[datasets[i]])
            ax.plot(range(len(this_replicate)), this_replicate, color=plot_color)
        
    
    
        #for j,this_peak_count_series in enumerate(series_peak_counts):
    
        #    #print(this_peak_count_series)
        #    print(plot_color)
        #    ax.scatter(range(len(this_peak_count_series)), this_peak_count_series, color=plot_color)
        #    ax.plot(range(len(this_peak_count_series)), this_peak_count_series, linestyle = faux_linestyle_cycle[j] % 4], color=plot_color)    
    return



def Get_One_Dataset(datadir, dataset, wavelength, file_identifier_strings):
    common_datapath = os.path.join(datadir, dataset)
    common_datapath = os.path.join(common_datapath, wavelength)
    #control_datapath = os.path.join(full_datapath, control_string)
    #perturb_string = os.path.join(full_datapath, perturb_string)

    #print(full_datapath)


    #print("hi")
    full_datapaths = []
    datapaths_identifiers = [] # identifies experimental condition within a study (i.e. perturbation vs. control)
    datapaths_repnumber = [] # replicate number
    
    for identifier in file_identifier_strings:
        #print("hi")
        identifier_subpath = os.path.join(common_datapath, identifier)
        subdirectories = [f.path for f in os.scandir(identifier_subpath) if f.is_dir()]
        for i,full_path in enumerate(subdirectories):
            full_datapaths.append(full_path)
            datapaths_identifiers.append(identifier)
            datapaths_repnumber.append(i)

    for full_path in full_datapaths:
        print(full_path)

    datafile_lists = []
    
    for full_path in full_datapaths:
        this_list = glob.glob(os.path.join(full_path, "**/*.csv"), recursive=True)
        this_list = sorted(this_list, key = sort_kavita_files)
        datafile_lists.append(this_list)

    # Extract data from datafiles and put them into a list of lists of pandas dataframes.
    # Each identifier string has its own list of dataframes associated with it
    data_value_series = []
    metadata_series = []   
   
    for i,this_identifier_string in enumerate(file_identifier_strings):
            #print(this_identifier_string)
            data_value_series.append([])
            metadata_series.append([])
    
    for i,this_datafile_list in enumerate(datafile_lists):
        
        for j,this_identifier_string in enumerate(file_identifier_strings):
            #print(this_identifier_string)
            if datapaths_identifiers[i] == this_identifier_string:
                data_value_series[j].append([])
                metadata_series[j].append([])
                for file in this_datafile_list:
                    #print(file)
                            data_value_series[j][-1].append( pd.read_csv(file, encoding="utf-16", sep="\\t", header=None, engine="python") )
                            data_value_series[j][-1][-1] = (data_value_series[j][-1][-1]).set_axis(["RT","Absorbance"],axis=1)
                            metadata_series[j][-1].append( os.path.basename(file) )

    return data_value_series, metadata_series

def Do_Water_Subtraction(raw_data_series, file_identifier_strings):
    #datafiles = glob.glob(os.path.join(full_datapath, "**/*.csv"), recursive=True)
    
    #datafiles = sorted(datafiles, key = sort_kavita_files)


    #if dataset == "KMSet9a":
    #    for file in datafiles:
    #        print(file)
    
    # Extract data from datafiles and put them into a list of lists of pandas dataframes.
    # Each identifier string has its own list of dataframes associated with it
    #data_value_series = []
    #metadata_series = []   
    
    #for i,this_identifier_string in enumerate(file_identifier_strings):
    #    #print(this_identifier_string)
    #    data_value_series.append([])
    #    metadata_series.append([])
    #    for file in datafiles:
    #        #print(file)
    #        if not fn.fnmatch(file, f'*{"rep2"}*'): # Hardcoded way to ignore the fact that KMSet9a and *ONLY* KMSet9a has a second replicate associataed with it
    #            if fn.fnmatch(file, f'*{this_identifier_string}*'):
    #                data_value_series[i].append( pd.read_csv(file, encoding="utf-16", sep="\\t", header=None, engine="python") )
    #                data_value_series[i][-1] = (data_value_series[i][-1]).set_axis(["RT","Absorbance"],axis=1)
    #                metadata_series[i].append( os.path.basename(file) )

    #return data_value_series, metadata_series

    #print(len(data_value_series[0]))
    
    
    # Subtract water from samples 
    data_series_subtracted = raw_data_series
    
    for i,this_identifier_string in enumerate(file_identifier_strings):
        for j,replicate in enumerate(data_series_subtracted[i]):
            for sample in replicate:
                sample["Absorbance"] = sample["Absorbance"] - data_series_subtracted[i][j][-1]["Absorbance"]

    return data_series_subtracted

def Get_Dataset_Peaks(data_series_subtracted, file_identifier_strings):
    # Get counts of peaks for each sample in each series
    series_peak_lists = []
    series_peak_counts = []
    for i,this_identifier_string in enumerate(file_identifier_strings):
        #control_peak_counts = []
        series_peak_lists.append([])
        series_peak_counts.append([])

        for j,this_replicate in enumerate(data_series_subtracted[i]):
            series_peak_lists[i].append([])
            series_peak_counts[i].append([])

            for k,this_sample in enumerate(this_replicate):
            
                if k < len(data_series_subtracted[i][j]) - 1: # ignore the water run at the end of the list
                    this_peak_list, _ = find_peaks(this_sample["Absorbance"], prominence=15)
                    series_peak_lists[i][j].append(this_peak_list)
                    series_peak_counts[i][j].append( len(this_peak_list) )

    return series_peak_lists, series_peak_counts


def Cluster_Many_Datasets(datadir, datasets, wavelength, file_identifier_strings, faux_color_cycle):

  
    
    #print(faux_color_cycle)
    #faux_color_cycle = rainbow_colormap([0:len(datasets)])

    # Get the peak count numbers from each dataset

    dataset_peak_counts = []
    metadata_series = []
    
    for i,dataset in enumerate(datasets):
        this_peak_counts, this_metadata = Get_One_Dataset(datadir=datadir, dataset=dataset, wavelength=wavelength, file_identifier_strings = file_identifier_strings) #, plot_color = faux_color_cycle[i])
        dataset_peak_counts.append(this_peak_counts)
        metadata_series.append(this_metadata)
        
    global fig, ax, datasets_dict

    dataset_peak_counts = np.asarray(dataset_peak_counts)
    print(dataset_peak_counts)
    



    data_values_flat = np.vstack((dataset_peak_counts[:,0],dataset_peak_counts[:,1]))

    #print(data_values_flat)
    
    X = StandardScaler().fit_transform(data_values_flat)
    kmeans = KMeans(n_clusters=2)
    kmeans.fit(X)
    ypred = kmeans.predict(X)
    #db = DBSCAN(eps=2.5,min_samples=3).fit(X)
    #print(db.fit_predict(X))

    for i,this_series in enumerate(data_values_flat):
        #plot_color = faux_color_cycle[db.fit_predict(X)[i]]
        plot_color = faux_color_cycle[ypred[i]]

        if i < len(datasets):
            this_linestyle = ':'
        else:
            this_linestyle = '-'
        
        #print(db.fit_predict(X)[i])
        #print(ypred[i])
        print(plot_color)
        ax.scatter(range(len(this_series)), this_series, color = plot_color)
        ax.plot(range(len(this_series)), this_series, color = plot_color, linestyle = this_linestyle)
    
    return

    #faux_linestyle_cycle = ['--', '-', ':', '-.']

    #for i,series_peak_counts in enumerate(dataset_peak_counts):
    #    plot_color = faux_color_cycle[i]

        #print("The first series is: ")
        #print(series_peak_counts[0])
        #print("The second series is: ")
        #print(series_peak_counts[1])
        
    #    this_peak_ratio_series = np.asarray(series_peak_counts[1]) / np.asarray(series_peak_counts[0])

        #print(f'The ratio is:\n{this_peak_ratio_series}')

     #   ax.scatter(range(len(this_peak_ratio_series)), this_peak_ratio_series, color = plot_color, label = (datasets[i]) + ": " + datasets_dict[datasets[i]])
      #  ax.plot(range(len(this_peak_ratio_series)), this_peak_ratio_series, color=plot_color)

    
    
        #for j,this_peak_count_series in enumerate(series_peak_counts):
    
        #    #print(this_peak_count_series)
        #    print(plot_color)
        #    ax.scatter(range(len(this_peak_count_series)), this_peak_count_series, color=plot_color)
        #    ax.plot(range(len(this_peak_count_series)), this_peak_count_series, linestyle = faux_linestyle_cycle[j] % 4], color=plot_color)    
 #   return





In [ ]:
# NOTE: For HPLC-UV/Vis analysis, run this cell FIRST, then run any of the other cells of your choosing.




# 
datadir = input("Paste in the complete filepath of the Supplemental Data Folder, up to and including the folder's name: ")

os.chdir(datadir)
datadir = os.path.join(datadir, 'HPLC\\Raw HPLC')
#"C:\Users\adean\OneDrive\Desktop\Labwork\Molecular Memory Project\\Old_Kavita_HPLC_Data\\Raw HPLC"
print('Using this directory:', datadir)
wavelength = "210nm"
control_string = "45_control"
perturb_string = "85_per"
file_inclusion_strings = [["210nm"], ["45_control","85_per"], ["rep1"]]
datasets_dict = {"KMSet2a":"tg_glc", "KMSet2b":"tg_Gly", "KMSet2c":"glc_Gly", "KMSet3a":"tg_glc_Gly", "KMSet4a":"tg_glc_Gly_gly", "KMSet4b":"tg_glc_Gly_urea", "KMSet5a":"tg_glc_Gly_gly_glu", "KMSet5b":"tg_glc_Gly_urea_glu", "KMSet6a":"tg_glc_Gly_gly_urea_glu", "KMSet7a":"tg_glc_Gly_gly_urea_glu_dec", "KMSet7b":"tg_glc_Gly_gly_urea_rib_dec", "KMSet7c":"tg_glc_Gly_gly_urea_ery_dec", "KMSet8a":"tg_glc_Gly_gly_urea_glu_dec_mg", "KMSet9a":"tg_glc_Gly_gly_urea_glu_dec_mg_ade"}


#----------------------------------
# Delete the '#' in front of the line containing the list of datasets you wish to see analysis for.
#Leave all others with a '#' in front of their line.

#datasets = ["KMSet2b", "KMSet2c", "KMSet3a", "KMSet4a", "KMSet4b", "KMSet5a", "KMSet5b", "KMSet6a", "KMSet7a", "KMSet7b", "KMSet8a", "KMSet9a"]#, "KMSet2a"]
datasets = ["KMSet2a", "KMSet2b", "KMSet2c", "KMSet3a", "KMSet4a", "KMSet4b", "KMSet5a", "KMSet5b", "KMSet6a", "KMSet7a", "KMSet7b", "KMSet8a", "KMSet9a"]#, "KMSet2a"]
#datasets = ["KMSet2b", "KMSet2c", "KMSet3a", "KMSet4a", "KMSet4b", "KMSet5a"]
#datasets = ["KMSet5b", "KMSet6a", "KMSet7a", "KMSet7b", "KMSet8a", "KMSet9a"]
#datasets = ["KMSet5b", "KMSet6a", "KMSet7a"]
#datasets = ["KMSet7b", "KMSet8a", "KMSet9a"]
#datasets = ["KMSet9a"]
#datasets = ["KMSet3a"]
#datasets = ["KMSet7c"]

### To show the systems that only contain tg, glc, and/or Gly:
#datasets = ["KMSet2a", "KMSet2b", "KMSet2c", "KMSet3a"]
#------------------------------------

In [ ]:
### FOR NORMAL PLOTTING ---------

# PRODUCES SI FIGURE S1

rainbow_colormap = mpl.colormaps["rainbow"].resampled(len(datasets))
faux_color_cycle = [rainbow_colormap(x) for x in range(len(datasets))]
fig, ax = plt.subplots()
Plot_Many_Datasets(datadir=datadir, datasets=datasets, wavelength=wavelength, file_identifier_strings = [control_string, perturb_string], faux_color_cycle=faux_color_cycle)
ax.set_title(f'Number of Species Observed by HPLC/UV-Vis\nin Perturbed and Unperturbed Systems')
ax.set_ylim(0,35)
ax.set_xlabel("Cycle Number")
ax.set_ylabel(f'Peak Counts at 210nm')
plt.legend(bbox_to_anchor = (1.79,1.021), loc="upper right")
#plt.savefig("Figure_1a_Comparing_Peak_Counts_v2_adr.svg", bbox_inches = "tight")
plt.show()

In [ ]:
### FOR RATIOS -----------------
rainbow_colormap = mpl.colormaps["rainbow"].resampled(len(datasets))
faux_color_cycle = [rainbow_colormap(x) for x in range(len(datasets))]
fig, ax = plt.subplots()
Plot_Many_Dataset_Ratios(datadir=datadir, datasets=datasets, wavelength=wavelength, file_identifier_strings = [control_string, perturb_string], faux_color_cycle=faux_color_cycle)
ax.set_title(f'Ratio of Number of Species Observed by HPLC/UV-Vis\nin Perturbed and Unperturbed Systems')
ax.set_ylim(0.5,3.5)
ax.set_xlabel("Cycle Number")
ax.set_ylabel(f'Ratio of Perturbed/Unperturbed Peak Counts at 210nm')
plt.legend(bbox_to_anchor = (1.8,1.0), loc="upper right")
#plt.savefig("Figure_1c_Peak_Count_Ratios_v3_adr.svg", bbox_inches = "tight")
plt.show()

In [ ]:
### TO SHOW THE REPRODUCIBILITY OF THE PEAK COUNTING METHOD --- 
# NOTE: THIS IS THE ONLY HPLC DATASET WITH REPLICATES, SO FOR IT, THE 'datasets' VARIABLE IS HARDCODED

# PRODUCES SI FIGURE S2

rainbow_colormap = mpl.colormaps["rainbow"].resampled(len(datasets))
faux_color_cycle = [rainbow_colormap(x) for x in range(len(datasets))]
fig, ax = plt.subplots()
Plot_Many_Datasets(datadir=datadir, datasets=['KMSet9a'], wavelength=wavelength, file_identifier_strings = [control_string, perturb_string], faux_color_cycle=faux_color_cycle)
ax.set_title(f'Comparison of KM Set 9a Replicates')
ax.set_ylim(0,35)
ax.set_xlabel("Cycle Number")
ax.set_ylabel(f'Peak Counts at 210nm')
plt.legend(bbox_to_anchor = (1.79,1.021), loc="upper right")
#plt.savefig("Figure_S_Comparing_KMSet9a.svg", bbox_inches = "tight")
plt.show()

In [ ]:
### PLOTTING THE RAW CHROMATOGRAMS

# PRODUCES SI FIGURES S4 TO S16

rainbow_colormap = mpl.colormaps["rainbow"].resampled(len(datasets))
faux_color_cycle = [rainbow_colormap(x) for x in range(len(datasets))]
for dataset in datasets:
    fig, ax = plt.subplots(1, 2, figsize = (6.4*2*4,4.8*4))
    ax[0].set_ylim([-100,2200])
    ax[1].set_ylim([-100,2200])
    ax[0].tick_params(axis='both', labelsize=32)
    ax[1].tick_params(axis='both', labelsize=32)
    ax[0].set_xlabel('Retention time (min)', fontsize=32)
    ax[1].set_xlabel('Retention time (min)', fontsize=32)
    ax[0].set_ylabel('Absorbance', fontsize=32)
    ax[1].set_ylabel('Absorbance', fontsize=32)

    rainbow_colormap = mpl.colormaps["rainbow"].resampled(11)
    faux_color_cycle = [rainbow_colormap(x) for x in range(11)]
    Plot_One_Dataset_HPLC(datadir=datadir, dataset=dataset, wavelength=wavelength, file_identifier_strings = [control_string, perturb_string], faux_color_cycle=faux_color_cycle)
    #plt.suptitle(f'{dataset}: {datasets_dict[dataset]}', fontsize=64)
    #plt.savefig(f'SI_Memory_Figure_{dataset}_HPLC.svg', bbox_inches='tight')
    plt.show()

In [ ]:
# FOR LC-MS/MS ANALYSIS FIGURES, RUN THIS CELL FIRST, THEN RUN THE CELL BELOW

# NOTE: We adapted the output from Compound Discoverer's PCA into SI Figures S33 and S34, we did not perform the PCA in Python.

# Setup
import os
import numpy as np
import math
import pandas as pd
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
import itertools
import copy

import sqlite3
import io
import zipfile
import xml.etree.ElementTree as ET
import networkx as nx


#-----
matplotlib.rcParams['svg.fonttype'] = 'none'
#-----

# Function definitions
def normalize_peak_areas(oligo_df, series, mode="per_feature"):
    
    if type(series[0]) == list:
        if mode == "per_feature":
            # Collect all unique indices
            all_entries = [x for this_series in series for x in this_series]
            all_entries = list(set(all_entries))
            #print(all_entries)

            for i in range(oligo_df.shape[0]): # For each row of the datafreame...
                #print(all_entries)
                #print(oligo_df)
                #print(oligo_df.iloc[i,all_entries])
                oligo_df.iloc[i,all_entries] = ( oligo_df.iloc[i,all_entries] / max(oligo_df.iloc[i,all_entries]) ) # Normalize by max observed intensity in any of the samples
    else:
        if mode == "per_feature":
            for i in range(oligo_df.shape[0]): # For each row of the dataframe...
                oligo_df.iloc[i,series] =  ( oligo_df.iloc[i,series] / max(oligo_df.iloc[i,series]) ) # Normalize by maximum observed intensity

    return oligo_df


# For ANOVA of individual feature peak areas grouped by oligomer composition
# For each row of the dataframe, we want to find the maximum group area, then divide all the individual areas that we care about by that value.
# This is distinct from how normalize_peak_areas() only looks within the group areas.



#def plot_subroutine(cycle_list, oligo_df, numeric_in):

    

def plot_oligos(oligo_df, numeric_indices, cv_indices, cycle_list, group_names=None, plot_ylim=None, figure_filename=None):
         
   # for i in range(oligo_df.shape[0]): # For each row of the dataframe...
   #     oligo_df.iloc[i,numeric_indices] =  ( oligo_df.iloc[i,numeric_indices] / max(oligo_df.iloc[i,numeric_indices]) ) # Normalize by maximum observed intensity

    fig, ax = plt.subplots()

    if not plot_ylim == None:
        ax.set_ylim(plot_ylim[0],plot_ylim[1])

    
    if type(numeric_indices[0]) == list: # Support for having multiple dataseries    
        faux_linestyle_cycle = ['--', '-', ':', '-.']
        faux_color_cycle = ["red", "blue", "yellow", "green", "orange"]
        for j,this_index_set in enumerate(numeric_indices):
            for i in range(oligo_df.shape[0]):
                #ax.plot(cycle_list, oligo_df.iloc[i,this_index_set], color = faux_color_cycle[i % 5], linestyle = faux_linestyle_cycle[j % 4])

                if group_names != None:
                    legend_label_name = ", " + group_names[j]
                else:
                    legend_label_name = ""

            
                error_values = np.asarray(oligo_df.iloc[i,this_index_set]) * ( np.asarray(oligo_df.iloc[i,cv_indices[j]]) / 100 ) # Calculates standard deviation of each datapoint from the CV and the group mean
                ax.errorbar(cycle_list, oligo_df.iloc[i,this_index_set],yerr=error_values, color=faux_color_cycle[i%5], linestyle = faux_linestyle_cycle[j % 4],
                           label = ("RT: " + str(oligo_df.iloc[i, 2]) + legend_label_name ))

            
                ax.scatter(cycle_list,oligo_df.iloc[i,this_index_set], color = faux_color_cycle[i % 5])
    else:  
        for i in range(oligo_df.shape[0]):
            ax.plot(cycle_list, oligo_df.iloc[i,numeric_indices])
            ax.scatter(cycle_list, oligo_df.iloc[i,numeric_indices], label = ("RT: " + str(oligo_df.iloc[i, 2])))
        
            #clean_values = np.asarray(four_glc.iloc[i,numeric_indices]).astype('float64')
            #minimum_values_array = np.ones(clean_values.shape) * -20 # Make sure never to plot anything lower than -1
            #ax.scatter([0,1,3,10], np.maximum.reduce([np.log(clean_values), minimum_values_array]), label = ("RT: " + str(four_glc.iloc[i, 1])) ) 
    plt.legend()
    if not (figure_filename == None):
        plt.savefig(figure_filename)
    


def plot_oligos_SI(oligo_df, numeric_indices, cv_indices, replicate_counts, cycle_list, fig, ax, plot_color, group_name=None, plot_ylim=None):
         
   # for i in range(oligo_df.shape[0]): # For each row of the dataframe...
   #     oligo_df.iloc[i,numeric_indices] =  ( oligo_df.iloc[i,numeric_indices] / max(oligo_df.iloc[i,numeric_indices]) ) # Normalize by maximum observed intensity

    #global fig, ax

    if not plot_ylim == None:
        ax.set_ylim(plot_ylim[0],plot_ylim[1])

    
    if type(numeric_indices[0]) == list: # Support for having multiple dataseries    
        faux_linestyle_cycle = ['--', '-', ':', '-.']
        #faux_color_cycle = ["red", "blue", "yellow", "green", "orange"]
        for j,this_index_set in enumerate(numeric_indices):
            for i in range(oligo_df.shape[0]):
                #ax.plot(cycle_list, oligo_df.iloc[i,this_index_set], color = faux_color_cycle[i % 5], linestyle = faux_linestyle_cycle[j % 4])
                
                #if group_names != None:
                #    legend_label_name = ", " + group_names[j]
                #else:
                #    legend_label_name = ""


                # Calculates an estimate of the 95% confidence interval of each datapoint from the CV and the group mean.
                # Note: Compound Discoverer calculates CV values by dividing the corrected sample standard deviation of a group by the group mean.
                # This code rederives the corrected sample standard deviation by multiplying each CV by its group mean. Then it divides by the number of replicates in that group at that cycle.
                # This gives an estimate of the standard error.
                # It then multiplies this by 2, to give +/- 2 standard errors. This is the 95% confidence interval.
                error_values = np.asarray(oligo_df.iloc[i,this_index_set]) * ( np.asarray(oligo_df.iloc[i,cv_indices[j]]) / 100 ) * (1 / np.sqrt(replicate_counts[j]) ) * 2
                
                error_bar_plot = ax.errorbar(cycle_list, oligo_df.iloc[i,this_index_set],yerr=error_values, color=plot_color, linestyle = faux_linestyle_cycle[j % 4], capsize=5)#,
                        #   label = ("RT: " + str(oligo_df.iloc[i, 2]) + legend_label_name ))
                if j==1:
                    if i==0:
                        error_bar_plot.set_label(group_name)      
                    face_coloring = plot_color
                    #print('beep')
                    
                else:
                    face_coloring = 'none'

                
                ax.scatter(cycle_list,oligo_df.iloc[i,this_index_set], facecolors=face_coloring, edgecolors=plot_color)
    else:  
        for i in range(oligo_df.shape[0]):
            ax.plot(cycle_list, oligo_df.iloc[i,numeric_indices])
            ax.scatter(cycle_list, oligo_df.iloc[i,numeric_indices], label = ("RT: " + str(oligo_df.iloc[i, 2])))
        
            #clean_values = np.asarray(four_glc.iloc[i,numeric_indices]).astype('float64')
            #minimum_values_array = np.ones(clean_values.shape) * -20 # Make sure never to plot anything lower than -1
            #ax.scatter([0,1,3,10], np.maximum.reduce([np.log(clean_values), minimum_values_array]), label = ("RT: " + str(four_glc.iloc[i, 1])) ) 
    
    #if not (figure_filename == None):
    #    plt.savefig(figure_filename)



def Group_Oligos_By_Length(oligo_df):
    # DEPRECIATED DO NOT USE
    oligo_lengths = set(oligo_df["Oligo Length"][:])
    oligo_length_locations = {}
    oligos_by_length = {}
    
    for this_oligo_length in oligo_lengths:
        oligo_length_locations[this_oligo_length] = []
    
    for i in range(len(oligo_df)):
        oligo_length_locations[oligo_df["Oligo Length"][i]].append(i)
    
    for this_oligo_length in oligo_lengths:
        oligos_by_length[this_oligo_length] = oligo_df.iloc[oligo_length_locations[this_oligo_length],:].copy()

    return oligos_by_length, oligo_lengths

    
def Group_Oligos_By_Composition(oligo_df):
    # DEPRECIATED DO NOT USE
    oligo_compositions = set(oligo_df["Annotation (Tentative)"][:])
    oligo_composition_locations = {}
    oligos_by_composition = {}

    for this_oligo_composition in oligo_compositions:
        oligo_composition_locations[this_oligo_composition] = []

    for i in range(len(oligo_df)):
        oligo_composition_locations[oligo_df["Annotation (Tentative)"][i]].append(i)

    for this_oligo_composition in oligo_compositions:
        oligos_by_composition[this_oligo_composition] = oligo_df.iloc[oligo_composition_locations[this_oligo_composition,:]].copy()

    return oligos_by_composition, oligo_compositions

def Group_Oligos_By_Property(oligo_df, grouping_property):
    # Get unique values of the grouping_property while maintaining ordering
    oligo_property_list = list(dict.fromkeys(oligo_df[grouping_property][:]))
    
    print('Oligo property list: ', oligo_property_list)
    oligo_locations_per_property = {}
    oligos_by_property = {}

    for this_property_value in oligo_property_list:
        oligo_locations_per_property[this_property_value] = []

    for i in range(len(oligo_df)):
        #print(grouping_property)
        #print(i)
        #print(oligo_df)
        oligo_locations_per_property[oligo_df[grouping_property][i]].append(i)

    for this_property_value in oligo_property_list:
        #print(this_property_value)
        #print(oligo_locations_per_property)
        #print(oligo_locations_per_property[this_property_value])
        oligos_by_property[this_property_value] = oligo_df.iloc[oligo_locations_per_property[this_property_value],:].copy().reset_index(drop=True)

    
    return oligos_by_property, oligo_property_list






def Filter_MS_By_Property(mass_spec_df, filtering_condition, ID_whitelist=None):

    filter_keys = []

    for i in range(len(mass_spec_df)):

        #print(mass_spec_df.iloc[i,:])
        if filtering_condition(mass_spec_df.iloc[i,:]):
            filter_keys.append(i)
        elif ID_whitelist is not None:
            if mass_spec_df['ID'][i] in ID_whitelist:
                #print('AAAAAAAAAAAAAA')
                filter_keys.append(i)
    
    return mass_spec_df.iloc[filter_keys,:].copy().reset_index(drop=True)
    


def Get_Glycine_Count_From_Composition_Name(composition_name):
    # NOTE: Brittle function. Will break with compositions other than glc and Gly as the only options, or if Gly count is listed before glc count.
    
    if len(composition_name.split('Gly')) == 2:
        return int(composition_name.split('Gly')[1])
    else:
        return 0


def plot_oligos_SI_All(oligo_df,  numeric_indices, cv_indices, replicate_counts, cycle_list, group_names=None, plot_ylim=None, figure_filename_base=None):

    oligo_df = oligo_df.reset_index(drop=True)
    
    oligos_by_length, oligo_lengths = Group_Oligos_By_Property(oligo_df, grouping_property="Oligomer Length")

    faux_color_cycle = ["red", "orange", "yellow", "green", "blue"]
    
    for j,this_oligo_length in enumerate(oligo_lengths):

        print(this_oligo_length)
        fig, ax = plt.subplots()
        
        oligos_by_composition, oligo_compositions = Group_Oligos_By_Property(oligos_by_length[this_oligo_length], grouping_property="Annotation by m/z")
        
        oligo_compositions = sorted(oligo_compositions, key = Get_Glycine_Count_From_Composition_Name)
        
        print(f'Oligo categories of {this_oligo_length}-mers is {len(oligos_by_composition)}')
        #print(f'Oligo compositions: {oligo_compositions}')
            

        faux_colormap = matplotlib.colormaps['rainbow'].reversed().resampled(len(oligos_by_composition))
        faux_color_cycle = faux_colormap(np.linspace(0,1,len(oligos_by_composition)))
        print(faux_color_cycle)

        
        for i,this_oligo_composition in enumerate(oligo_compositions):
            #print(this_oligo_composition)

            #return oligos_by_composition[this_oligo_composition]

            oligo_abundances_normalized = normalize_peak_areas(oligos_by_composition[this_oligo_composition], series=numeric_indices)            

            plot_oligos_SI(oligo_abundances_normalized, numeric_indices, cv_indices, replicate_counts, cycle_list, fig=fig, ax=ax, plot_color=faux_color_cycle[i], group_name=this_oligo_composition, plot_ylim=plot_ylim)

        ax.set_ylabel('Normalized Feature Abundance')
        ax.set_xlabel('Cycle Number')
        plt.legend()
        #plt.title(f'{this_oligo_length}-mers')


        if not (figure_filename_base==None):
            plt.savefig(figure_filename_base + f'_{this_oligo_length}-mers' + '.svg', bbox_inches = 'tight')



def plot_oligos_by_Composition(oligo_df, target_oligomer_length,  numeric_indices, cv_indices, replicate_counts, cycle_list, group_names=None, plot_ylim=None, figure_filename_base=None):

    oligo_df = oligo_df.reset_index(drop=True)
    
    oligos_by_length, oligo_lengths = Group_Oligos_By_Property(oligo_df, grouping_property="Oligomer Length")

    #faux_color_cycle = ["red", "orange", "yellow", "green", "blue"]

    #return oligos_by_length
    #print('Oligo lengths:', oligo_lengths)

    #print(oligos_by_length[7])
    
    oligos_by_composition, oligo_compositions = Group_Oligos_By_Property(oligos_by_length[target_oligomer_length], grouping_property="Annotation by m/z")

    print('Oligo compositions: ',oligo_compositions)
    
    #print(f'Oligo categories of {this_oligo_length}-mers is {len(oligos_by_composition)}')

    #faux_colormap = matplotlib.colormaps['rainbow'].reversed().resampled(len(oligos_by_composition))
    #faux_color_cycle = faux_colormap(np.linspace(0,1,len(oligos_by_composition)))
    #print(faux_color_cycle)

    
    for i,this_oligo_composition in enumerate(oligo_compositions):
        
        fig, ax = plt.subplots()
        #print(this_oligo_composition)

        #return oligos_by_composition[this_oligo_composition]

        oligo_abundances_normalized = normalize_peak_areas(oligos_by_composition[this_oligo_composition], series=numeric_indices)            

        plot_oligos_SI(oligo_abundances_normalized, numeric_indices, cv_indices, replicate_counts, cycle_list, fig=fig, ax=ax, plot_color='black', group_name=this_oligo_composition, plot_ylim=plot_ylim)
    
        plt.legend()
        plt.title(f'{this_oligo_composition}')

        if not (figure_filename_base==None):
            plt.savefig(figure_filename_base + f'_{this_oligo_composition}' + '.svg')


def Mass_From_Composition(composition, monomer_list, ion_type='[M-H]-'):

    #composition = {'glycine': 5, 'glycolic_acid':2}

    #print(composition)
    
    mass_1H = 1.0078250319
    mass_12C = 12
    mass_14N =14.003074
    mass_16O = 15.99491462
    mass_32S = 31.97207117
    mass_electron = 5.49E-04

    mass_dictionary = {}

    mass_dictionary['water'] = (2*mass_1H) + (1*mass_16O)
    
    mass_dictionary['glycine'] = (2*mass_12C) + (5*mass_1H) + (1*mass_14N) + (2*mass_16O)

    mass_dictionary['glycolic_acid'] = (2*mass_12C) + (4*mass_1H) + (0*mass_14N) + (3*mass_16O)

    mass_dictionary['alanine'] = (3*mass_12C) + (7*mass_1H) + (1*mass_14N) + (2*mass_16O)

    mass_dictionary['valine'] = (5*mass_12C) + (11*mass_1H) + (1*mass_14N) + (2*mass_16O)

    compound_mass_uncombined = 0.0
    for i,this_monomer in enumerate(monomer_list):
        #print(this_monomer)
        compound_mass_uncombined += composition[i] * mass_dictionary[monomer_list[i]]

    oligomer_length = np.sum(composition)

    oligomer_neutral_mass = compound_mass_uncombined - ((oligomer_length-1)*(mass_dictionary['water']))

    if (ion_type == '[M-H]-'):
        oligomer_mz = (oligomer_neutral_mass - mass_1H) + mass_electron
        return oligomer_mz

    return


def Enumerate_Compositions(oligomer_length, monomer_list):

    composition_list = np.zeros([math.comb(len(monomer_list)+oligomer_length-1, oligomer_length), len(monomer_list)])

    enumeration_elements = np.identity(len(monomer_list)) # List each possible monomer as a unit vector

    for i,this_composition in enumerate(itertools.combinations_with_replacement(enumeration_elements, oligomer_length)):
        composition_list[i] = sum(this_composition) # Find this composition in a space defined by each monomer as an orthogonal unit vector
        # composition_list[i]'s values now correspond to how many of each monomer residue are present in this composition.

    return composition_list

        
def Composition_To_Name(composition, monomer_list):

    monomer_abbreviation_dict = {'glycolic_acid' : 'glc', 'glycine' : 'Gly', 'valine' : 'Val', 'alanine' : 'Ala' }

    composition_name = ''
    
    for i,this_monomer_amount in enumerate(composition):
        if this_monomer_amount != 0:
            composition_name = composition_name + monomer_abbreviation_dict[monomer_list[i]] + str(int(this_monomer_amount))

    return composition_name

def Name_To_Composition(composition_name, monomer_list=['glycolic_acid','glycine']):

    raise NotImplementedError()
    
    monomer_abbreviation_dict = {'glycolic_acid' : 'glc', 'glycine' : 'Gly', 'valine' : 'Val', 'alanine' : 'Ala' }

    for monomer_type in monomer_list:
        if composition_name.startswith(monomer_abbreviation_dict[monomer_type]):
            pass
    
    return


def Make_Composition_List(oligomer_max_length, monomer_list):

    print('Starting to Make Composition List')
    last_composition_list = None
    composition_list = None  
    for i in range(1,oligomer_max_length+1):

        this_composition_list = Enumerate_Compositions(i, monomer_list)

        if last_composition_list is None:
            composition_list = this_composition_list
            #print(composition_list)
        else:
            #print(last_composition_list)
            #print(this_composition_list)
            composition_list = np.concatenate([composition_list, this_composition_list])
            #print(i, composition_list)

        last_composition_list = this_composition_list.copy()
        #print(last_composition_list.base is None)

    print('Finished Making Composition List')

    return composition_list

def Make_Mass_List(composition_list, monomer_list):
    print('Starting to Make Mass List')

    mass_list = np.zeros(len(composition_list),dtype='float64')
    mass_to_name_dict = {}
    
    for i,this_composition in enumerate(composition_list):
        mass_list[i] = Mass_From_Composition(this_composition, monomer_list)

        mass_to_name_dict[mass_list[i]] = Composition_To_Name(this_composition, monomer_list) 
        
        
    print('Finished Making Mass List')
    return mass_list, mass_to_name_dict

#def Mass_To_Composition():
#    pass


def Check_Mass_Match_Debug(theoretical_mass, measured_mass, mz_tolerance_threshold=3, mode="ppm"):

    if mode == "ppm":
        print('Theoretical mass:', theoretical_mass)
        print('Measured mass:', measured_mass)
        ppm_difference = ((measured_mass - theoretical_mass) / theoretical_mass) * 1000000
        print(ppm_difference)
        return np.abs(ppm_difference) < mz_tolerance_threshold
        
    elif mode == "absolute":
        return np.abs(theoretical_mass - measured_mass) < mz_tolerance_threshold

def Check_Mass_Match(theoretical_mass, measured_mass, mz_tolerance_threshold=3, mode="ppm"):

    if mode == "ppm":
        #print('Theoretical mass:', theoretical_mass)
        #print('Measured mass:', measured_mass)
        ppm_difference = ((measured_mass - theoretical_mass) / theoretical_mass) * 1000000
        #print(ppm_difference)
        return np.abs(ppm_difference) < mz_tolerance_threshold
        
    elif mode == "absolute":
        return np.abs(theoretical_mass - measured_mass) < mz_tolerance_threshold
        


def Extract_Oligomers_From_Mass_Spectrometry_Data(mass_spec_df, mz_tolerance, mode, oligomer_max_length, monomer_list, ID_whitelist=None):

    composition_list = Make_Composition_List(oligomer_max_length, monomer_list)
    print(f'Composition list:\n{composition_list}')
    
    mass_list, mass_to_name_dict = Make_Mass_List(composition_list, monomer_list)
    print(f'Mass list:\n{mass_list}')


    print('Starting to Filter by m/z')
    filter_keys = []
    mass_annotation_list = []
    oligomer_length_list = []

    for i in range(len(mass_spec_df)):
        for j,this_mass in enumerate(mass_list):
            if Check_Mass_Match(this_mass, mass_spec_df['m/z'][i], mz_tolerance, mode=mode):
                filter_keys.append(i)
                mass_annotation_list.append(mass_to_name_dict[this_mass])
                oligomer_length_list.append(int(sum(composition_list[j])))
                break
            elif ID_whitelist is not None:
                if mass_spec_df['ID'][i] in ID_whitelist:
                    filter_keys.append(i)
                    oligomer_length_list.append('N/A - Whitelisted')
                    break
    #print(filter_keys)
    #print(len(mass_spec_df))
    oligomer_df = mass_spec_df.iloc[filter_keys,:].copy().reset_index(drop=True)

    oligomer_df["Annotation by m/z"] = mass_annotation_list
    oligomer_df['Oligomer Length'] = oligomer_length_list
    
    print('Done Filtering by m/z')
    return oligomer_df

        
def Clean_Oligo_Data(oligo_df, ID_whitelist=None):

    has_MS2 = lambda a : a['MS2'] == 'DDA'

    oligo_df_filtered = Filter_MS_By_Property(oligo_df, has_MS2, ID_whitelist)

    decent_peak_quality = lambda a : a['Peak Rating (Max.)'] >= 7

    oligo_df_filtered = Filter_MS_By_Property(oligo_df_filtered, decent_peak_quality, ID_whitelist)
    
    
    
    
    
    
    #filtering_condition = lambda a : a[]

    return oligo_df_filtered


def Get_Feature_MS2_Spectrum(feature_ID, result_file_path='AD112_KM184_Perturbation_Memory.cdResult'):
    #print(result_file_path)
    conn = sqlite3.connect(result_file_path)
    
    cur = conn.cursor()


    # This code has been adapted from a PFAS analysis script by Ralf Tautenhahn
    
    # The MassSpectrumInfoItems table contains a list of IDs for mass spectra, both MS1 and MS2 spectra.
    # The ConsolidataedUnknownCompoundIonItems table is the feature table.
    # The BestHitIonInstanceItems table is a list of all the most representataive ions for a given feature/compound.
    # The BestHitIonInstanceItemsMassSpectrumInfoItems is a "connected table" that links the BestHitIonInstanceItems table and the MassSPectrumInfoItems table.
    # These connected tables are what are used to link the MassSpectrumInfoItemsID to a given best-match ion and that best-match ion to a given feature or compound.

    # In plain English, the first query looks for an entry in the Mass Spectrum ID table such that:
    #    1. The best-hit ion ID is the same in the connected table between best-hit ions and mass spectra IDs as it is in the connected table between features and best-hit ions.
    #    2. The feature ID is the same in the features table as it is in the connected table between features and best-hit ions.
    #    3. The best-hit ion ID is the samae in the best-hit ions table as it is in the connected table between features and best-hit ions.
    #    4. The best-hit type in the best-hit ion table is 2, meaning that it is for an MS2 spectrum.
    #    5. The feature ID matches the one that has been passed into this function.

    
    # The compound table can be accessed as ConsolidatedUnknownCompoundItems, rather than ConsolidatedUnknownCompoundIonItems for the feataure table.
    # Removing the "Ion" part from this table and where it appears in connected tables in the query will allow you to find the MS2 spectra of compounds instead of features
    # But we aren't doing that since the feature->compound filtering is excluding a lot of data that we care about.

    # BestHitIonInstanceItems.BestHitType describes if the best hit ion is for an MS1 or an MS2 spectrum.
    # Setting it to 1 in the query instead of 2 would give the best MS1 spectrum for a feature.
    # But that would break the rest of the code in this function (GEt_Feature_MS2_Spectrum) since it assumes it gets MS2 data,
    # and MS2 data actually consists of both an MS1 scan and the follow-up MS2 scan. This MS1 scan is typically not the same as the best MS1 scan,
    # so the best MS1 scan should be accessed directly if needed.
 
    
    query = """SELECT MassSpectrumInfoItemsID
                      FROM BestHitIonInstanceItemsMassSpectrumInfoItems, ConsolidatedUnknownCompoundIonItemsBestHitIonInstanceItems, ConsolidatedUnknownCompoundIonItems, BestHitIonInstanceItems
                      WHERE BestHitIonInstanceItemsMassSpectrumInfoItems.BestHitIonInstanceItemsID = ConsolidatedUnknownCompoundIonItemsBestHitIonInstanceItems.BestHitIonInstanceItemsID
                      AND ConsolidatedUnknownCompoundIonItems.ID = ConsolidatedUnknownCompoundIonItemsBestHitIonInstanceItems.ConsolidatedUnknownCompoundIonItemsID
                      AND BestHitIonInstanceItems.ID = ConsolidatedUnknownCompoundIonItemsBestHitIonInstanceItems.BestHitIonInstanceItemsID
                      AND BestHitIonInstanceItems.BestHitType = 2
                      AND ConsolidatedUnknownCompoundIonItems.ID =""" + str(feature_ID)
    
    cur.execute(query)
    query_response = cur.fetchall()
    
    if len(query_response) < 2:
        raise ValueError(f'Data missing for MS/MS spectra with feature ID {feature_ID}')
    if len(query_response) > 2:
        raise ValueError(f'Expected one parent MS1 spectrum and one associated MS2 spectrum. Got {len(query_response)} results instead of 2.')
    
    query_2 = "SELECT Spectrum FROM MassSpectrumItems WHERE ID=" + str(query_response[0][0])
    cur.execute(query_2)
    local_MS1_spectrum = cur.fetchall()
    
    query_3 = "SELECT Spectrum FROM MassSpectrumItems WHERE ID=" + str(query_response[1][0])
    cur.execute(query_3)
    best_MS2_spectrum = cur.fetchall()
    
    conn.close()

    return local_MS1_spectrum[0][0], best_MS2_spectrum[0][0] # Want to return bytes objects, not the length-1 list of length-1 tuple we get from fetchall()


def Decode_Spectrum_XML_Binary(spectrum_binary):
    xml_tree = None
    xml_root = None

    buffer = io.BytesIO(spectrum_binary)
    with zipfile.ZipFile(buffer, 'r') as xmlzip:
        #for x in xmlzip.infolist():
        #    print("aaa")
        #    print(x)

        xml_file_data = xmlzip.read(xmlzip.infolist()[0])

        xml_tree = ET.parse(io.BytesIO(xml_file_data))
        xml_root = xml_tree.getroot()

    #print(xml_tree)
    return xml_tree, xml_root

def Make_Mass_Spectrum_Array(spectrum_xml_root):

    peaks = spectrum_xml_root.find('PeakCentroids')

    peak_array = np.zeros([2,len(peaks)])
    print(len(peaks))
    
    for i,peak in enumerate(peaks):
        peak_array[0][i] = peak.attrib['X']
        peak_array[1][i] = peak.attrib['Y']
        #print(peak.attrib['Y'])
    return peak_array


def Annotate_Peaks(peak_array, parent_ion_name, parent_ion_length, mass_list, mass_to_name_dict, composition_list, mz_tolerance, mode='ppm'):

    print("Annotating peaks...")
    spectrum_annotation_list = []
    is_major_annotation = []


    
    
    for i, this_peak_mass in enumerate(peak_array[0]):
        spectrum_annotation_list.append(str(this_peak_mass))

        is_major_annotation.append(False)

        
        for j, this_library_mass in enumerate(mass_list):
            if Check_Mass_Match(this_library_mass, this_peak_mass, mz_tolerance, mode=mode):
                is_major_annotation[-1] = True
                composition_name = mass_to_name_dict[this_library_mass]
                oligomer_length = (int(sum(composition_list[j])))

                if composition_name == parent_ion_name and oligomer_length == parent_ion_length:
                    annotation_name = spectrum_annotation_list[-1] + '\n' + f'Precursor: [{composition_name}-H]-'
                    spectrum_annotation_list[-1] = annotation_name
                else:
                    # Note that annotation as of now assumes that we're only dealing with [M-H]-, y-series ions 
                    annotation_name = spectrum_annotation_list[-1] + '\n' + f'y{oligomer_length}: [{composition_name}-H]-'
                    spectrum_annotation_list[-1] = annotation_name
                break             
    print("Finished annotating peaks.")
    return spectrum_annotation_list, is_major_annotation

def is_Colliding(displacement_vector):
    has_self_collisions = (np.abs(displacement_vector[0]) < 6.5) and (np.abs(displacement_vector[1]) < 2.7)
    return has_self_collisions

def Get_Parent_Ion(feature_ID, oligomer_df):

    parent_ion_name = ''
    parent_ion_length = ''

    for i in range(len(oligomer_df)):

        #print(mass_spec_df.iloc[i,:])
        if oligomer_df.iloc[i,:]['ID'] == feature_ID:
            parent_ion_name = oligomer_df.iloc[i,:]['Annotation by m/z']
            parent_ion_length = oligomer_df.iloc[i,:]['Oligomer Length']

    if parent_ion_name == '' or parent_ion_length == '':
        raise Error()
    
    return parent_ion_name, parent_ion_length



def repel_labels(ax, x, y, labels, k=0.01):
    # Adapted from - https://stackoverflow.com/a/34697108
    # Posted by unutbu, modified by community. See post 'Timeline' for change history
    # Retrieved 2026-08-14, License - CC BY-SA 3.0
    
    G = nx.DiGraph()
    data_nodes = []
    init_pos = {}

    max_x = math.ceil(max(x))
    min_x = math.floor(min(x))

    floor_range = np.linspace(min_x, max_x, int((max_x - min_x)*0.4))

    for floor_position in floor_range:
        #print('floor',floor_position)
        data_str = f'floor_{floor_position}'
        G.add_node(data_str)
        data_nodes.append(data_str)
        init_pos[data_str] = (floor_position, -50)
    
    for xi, yi, label in zip(x, y, labels):
        data_str = 'data_{0}'.format(label)
        G.add_node(data_str)
        G.add_node(label)
        G.add_edge(label, data_str)    
        data_nodes.append(data_str)
        init_pos[data_str] = (xi, yi)
        init_pos[label] = (xi, yi)

        

    pos = nx.spring_layout(G, pos=init_pos, fixed=data_nodes, k=k)

    # undo spring_layout's rescaling
    pos_after = np.vstack([pos[d] for d in data_nodes])
    pos_before = np.vstack([init_pos[d] for d in data_nodes])
    scale, shift_x = np.polyfit(pos_after[:,0], pos_before[:,0], 1)
    scale, shift_y = np.polyfit(pos_after[:,1], pos_before[:,1], 1)
    shift = np.array([shift_x, shift_y])
    for key, val in pos.items():
        pos[key] = (val*scale) + shift

    for edge_label, data_str in G.edges():
        while pos[edge_label][1] < 0:
            pos[edge_label] = (pos[edge_label][0], pos[edge_label][1]+1)
            
    text_positions = []
    for edge_label, data_str in G.edges():
        for list_label in labels:
            if list_label == edge_label:
                text_positions.append(pos[edge_label])
                break
    return text_positions
    
            

        
        
   #     print('position:', pos[label])
   # 
   #         
   # for label, data_str in G.edges():
   #     if 
   #     ax.annotate(label,
   #                 xy=pos[data_str], xycoords='data',
   #                 xytext=pos[label], textcoords='data',
   #                 arrowprops=dict(arrowstyle="-",
   #                                 shrinkA=0, shrinkB=0,
   #                                 connectionstyle="arc3", 
   #                                 color='gray'), color='gray' )
    # expand limits
    #all_pos = np.vstack(list(pos.values()))
    #x_span, y_span = np.ptp(all_pos, axis=0)
    #mins = np.min(all_pos-x_span*0.15, 0)
    #maxs = np.max(all_pos+y_span*0.15, 0)
    #ax.set_xlim([mins[0], maxs[0]])
    #ax.set_ylim([mins[1], maxs[1]])
    #ax.set_ylim(0)

#fig, ax = plt.subplots()
#ax.scatter(scatter_data[:, 0], scatter_data[:, 1],
#           c=scatter_data[:, 2], s=scatter_data[:, 2] * 150)
#labels = ['ano_{}'.format(i) for i in range(N)]
#repel_labels(ax, scatter_data[:, 0], scatter_data[:, 1], labels, k=0.008)

#plt.show()


    
def Plot_MS2_Spectrum(feature_ID, oligomer_df, cdresult_path='AD112_KM184_Perturbation_Memory.cdResult', mz_tolerance=3, mode='ppm', oligomer_max_length=15, monomer_list=['glycolic_acid', 'glycine']):

    print('Starting plot process for feature ID ', feature_ID)
    
    _, MS2_spectrum_binary = Get_Feature_MS2_Spectrum(feature_ID, cdresult_path)

    _, spectrum_xml_root = Decode_Spectrum_XML_Binary(MS2_spectrum_binary) 

     
    # make array
    peak_array = Make_Mass_Spectrum_Array(spectrum_xml_root)

    #normalize
    peak_array[1] = (peak_array[1]/max(peak_array[1])) * 100

    
    

    # plot the lines

    fig, ax = plt.subplots(figsize = (6.4*4,4.8*4))
    ax.set_ylim([0,110])
    
    ax.vlines(peak_array[0], 0, peak_array[1], color="black")

    # annotate

    
    
    composition_list = Make_Composition_List(oligomer_max_length, monomer_list)
    #print(f'Composition list:\n{composition_list}')
    
    mass_list, mass_to_name_dict = Make_Mass_List(composition_list, monomer_list)
    #print(f'Mass list:\n{mass_list}')

    feature_parent_ion_name, feature_parent_ion_length = Get_Parent_Ion(feature_ID, oligomer_df)
    
    spectrum_annotation_list, is_major_annotation = Annotate_Peaks(peak_array, feature_parent_ion_name, feature_parent_ion_length, mass_list, mass_to_name_dict, composition_list, mz_tolerance, mode=mode)

    #for 
    
    #print(peak_array)
    text_positions = np.asarray(repel_labels(ax, x=peak_array[0,:], y=peak_array[1,:], labels=spectrum_annotation_list, k=2))
    
    finalized_text_positions = []
    for i, annotation in enumerate(spectrum_annotation_list):
        if is_major_annotation[i]:
            this_location = np.asarray([peak_array[0][i], peak_array[1][i]])
            ax.annotate(annotation,
                    xy=this_location, xycoords='data',
                    xytext=text_positions[i], textcoords='data',
                    arrowprops=dict(arrowstyle="-",
                                    shrinkA=0, shrinkB=0,
                                    connectionstyle="arc3", 
                                    color='gray'),
                    color='black',
                    fontsize=15)
            finalized_text_positions.append(text_positions[i])
    for i, annotation in enumerate(spectrum_annotation_list):
        if not is_major_annotation[i]: 
            displacements =  np.asarray([text_positions[i] - np.asarray(x) for x in finalized_text_positions])
            #print('displacements:', displacements)
            if not any([is_Colliding(x) for x in displacements]):
                this_location = np.asarray([peak_array[0][i], peak_array[1][i]])
                ax.annotate(annotation,
                        xy=this_location, xycoords='data',
                        xytext=text_positions[i], textcoords='data',
                        arrowprops=dict(arrowstyle="-",
                                        shrinkA=0, shrinkB=0,
                                        connectionstyle="arc3", 
                                        color='gray'),
                        color='gray')
            

    

    scan_number = spectrum_xml_root.find('Header').find('SpectrumIdentifiers').find('SpectrumIdentifier').attrib['ScanNumber']
    isolation_mass = spectrum_xml_root.find('PrecursorInfo').attrib['IsolationMass']
    retention_time = '{rt:.3f}'.format(rt=float(spectrum_xml_root.find('Header').find('SpectrumIdentifiers').find('SpectrumIdentifier').attrib['RetentionTime']))

    
    
    plot_information = f'Scan #{scan_number}\nIsolation Mass: {isolation_mass}\nRetention Time: {retention_time}'
    ax.text(0.85, 0.98, plot_information, transform=ax.transAxes,
        verticalalignment='top', fontsize=15)
    
    #plt.savefig(f'{feature_parent_ion_length}-mer_MS2_feature_{feature_ID}_composition_{feature_parent_ion_name}.svg')
    return
    #annotation_positions_list = []
    
    #for i,annotation in enumerate(spectrum_annotation_list):
    #    if annotation != '':   
    #        #print(annotation)
#
 #           this_location = np.asarray([peak_array[0][i], peak_array[1][i]])
  #          next_annotation_position = np.asarray([this_location[0], this_location[1]+1])
#
 #           if not annotation_positions_list == []:
  #              #print(annotation_positions_list)
   #             annotation_displacements = np.asarray([next_annotation_position - np.asarray(x) for x in annotation_positions_list])
    #            #print(type(annotation_displacements))
     #           #print(annotation_displacements[0].shape)
      #          annotation_distances = np.asarray([np.linalg.norm(x) for x in annotation_displacements])
#
 #               #print('Annotation displacements:', annotation_displacements[:,1])
#
 #               condition_A = min(annotation_displacements[:,1]) < 1.5
  #              condition_B = min(annotation_displacements[:,0]) < 5
   #             #print(condition_A)
    #            #print(condition_B)
#
 #               repetitions_count = 0
  #              while (any([is_Colliding(x) for x in annotation_displacements])):
#
 #                   #print('we reach here')
  #                  
   #                 #print('a')
    #                force_sum = np.zeros(2)
#
 #                   for i in range(len(annotation_displacements)):
  #                      force_sum += annotation_displacements[i] / np.power(annotation_distances[i],2)
   #                     
    #                force_sum = force_sum * (1 + repetitions_count/20)
#
 #                   if next_annotation_position[1] < 3:
  #                      force_sum += np.asarray([0, 1 + repetitions_count/20])
#
 #                   #print('force sum:', force_sum)
  #                  next_annotation_position += 0.4*force_sum
   #                 
    #                annotation_displacements = np.asarray([next_annotation_position - np.asarray(x) for x in annotation_positions_list])   
     #               annotation_distances = np.asarray([np.linalg.norm(x) for x in annotation_displacements])
#
 #                   #print(type(annotation_displacements))
  #                  
   #                 repetitions_count += 1
    ##                   print('Failed to find non-collding solution for annotation positions')
      #                  break
       #         #print(f'Repetition count for {annotation}:', repetitions_count)
                

   #         ax.annotate(annotation, this_location, next_annotation_position, arrowprops=dict(arrowstyle='-',connectionstyle="Arc3,rad=0.0007", color='gray'), color='gray')
   #         annotation_positions_list.append(next_annotation_position)
   #         #annotation_positions_list.append(next_annotation_position*2)
   #
   
    



In [ ]:
# AFTER SETUP, RUN THIS CELL, THEN RUN ANY OF THE CELLS BELOW

shared_file_path = input("Paste in the complete filepath of the Supplemental Data Folder, up to and including the folder's name: ")
os.chdir(shared_file_path)
compound_discoverer_path = os.path.join(shared_file_path, 'LC-MS2\\Compound Discoverer\\Compound_Discoverer_AD112_KM184_Perturbation_Memory_All_Features_All_Columns.csv')

compound_discoverer_export = pd.read_csv(compound_discoverer_path)

cleaned_table = Extract_Oligomers_From_Mass_Spectrometry_Data(compound_discoverer_export, 3, "ppm", 15, ['glycolic_acid', 'glycine'])
ID_whitelist = [6028, 141] # ID 6028 is the glc1Gly6 feature.
# ID 141 is the most intense (by far) Gly1 feature.

filtered_table = Clean_Oligo_Data(cleaned_table, ID_whitelist)
filtered_table = filtered_table.sort_values(by=['Oligomer Length', 'Annotation by m/z', 'ID'])
feature_blacklist_filepath = os.path.join(shared_file_path, 'LC-MS2\\Compound Discoverer\\AD112_KM184_Feature_Blacklist.csv')
feature_blacklist = pd.read_csv(feature_blacklist_filepath)
not_in_blacklist = lambda a : (not (a["ID"] in list(feature_blacklist["ID"])))
filtered_table_ppm = Filter_MS_By_Property(filtered_table, not_in_blacklist, ID_whitelist)

In [ ]:
# ABUNDANCE GRAPHS, PRODUCES FIGURE 2 IN THE MAIN MANUSCRIPT

numeric_indices = [[73,74,75,76,77],[73,78,79,80,81]]
cv_indices=[[82,83,84,85,86],[82,87,88,89,90]]

plot_oligos_by_Composition(oligo_df=filtered_table_ppm, target_oligomer_length=7,  numeric_indices=numeric_indices, cv_indices=cv_indices, replicate_counts=[[2,3,3,3,3],[2,4,4,4,4]], cycle_list=[0,1,3,6,10], group_names=['Control','Perturb'], plot_ylim=[-0.1,1.6], figure_filename_base='Heptamer_Features_Deduplicated')

In [ ]:
# ABUNDANCE GRAPHS. PRODUCES SI FIGURES S35 TO S40

plot_oligos_SI_All(filtered_table_ppm.copy(), numeric_indices = [[73,74,75,76,77],[73,78,79,80,81]], cv_indices=[[82,83,84,85,86],[82,87,88,89,90]], replicate_counts = [[2,3,3,3,3],[2,4,4,4,4]],  cycle_list = [0,1,3,6,10],
    group_names = ["Control","Perturb"], plot_ylim=[-0.1,1.6], figure_filename_base='SI_Memory_Figure_Other_Oligos_v3')#"SI_Memory_Figure_Other_Oligos_v2")

In [ ]:
# MS/MS SPECTRA. PRODUCES SI FIGURES S17 TO S32, among other unincluded feature MS/MS spectra.

for i in range(len(filtered_table_ppm)):
    if filtered_table_ppm.iloc[i,:]['ID'] not in ID_whitelist: # We exclude whitelisted feature IDs because they don't have MS/MS spectra
        Plot_MS2_Spectrum(filtered_table_ppm.iloc[i,:]['ID'], filtered_table_ppm, cdresult_path=str(os.path.join(shared_file_path, 'LC-MS2\\Compound Discoverer\\AD112_KM184_Perturbation_Memory.cdResult')))